# E16 — O que foi apagado

O capítulo anterior mediu o mundo que não foi observado. Este mede o espelho dele: o que o registro
**teve** e já não tem, porque a memória do sistema foi jogada fora. A pergunta é se há preço que
traga o passado de volta.

O sistema é o mais simples que esquece: x de hoje é uma fração a do x de ontem mais ruído novo. Com
a perto de um, a memória é longa; com a perto de zero, o processo esquece tudo num passo. A
tentativa que vem à cabeça é desfazer o decaimento: dividir o estado de hoje por a, uma vez para
cada dia que se quer andar para trás.

Duas reconstruções entram na balança. A **inversa** divide pelo decaimento e não sabe o que fazer
com o ruído que entrou em cada passo. A **ótima** é a esperança condicional: dado o estado de hoje,
o melhor palpite para k dias atrás é o estado de hoje encolhido por a elevado a k. As duas têm
erro em forma fechada, e é isso que o caderno mede.


In [1]:
# <- brinque com: VALORES_DE_A, DIAS, SEMENTES, TOLERANCIAS, BITS
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import esquecimento, graficos, mudanca

VALORES_DE_A = (0.5, 0.8, 0.9, 0.95, 0.99)
DIAS = 20000
SEMENTES = 12
TOLERANCIAS = (0.5, 0.9)
BITS = (4, 8, 16, 53)
GRADE = (1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144)
SIGMA = 1.0
SEMENTE = 400

print("a escala do processo e sigma sobre raiz de um menos a ao quadrado")
for a in VALORES_DE_A:
    print("  a %.2f | escala %.4f | memoria 1/(1-a) %.1f dias | horizonte a 50%%: %.1f | a 90%%: %.1f"
          % (a, SIGMA / np.sqrt(1 - a ** 2), 1 / (1 - a), esquecimento.horizonte(a, 0.5),
             esquecimento.horizonte(a, 0.9)))


a escala do processo e sigma sobre raiz de um menos a ao quadrado
  a 0.50 | escala 1.1547 | memoria 1/(1-a) 2.0 dias | horizonte a 50%: 0.2 | a 90%: 1.2
  a 0.80 | escala 1.6667 | memoria 1/(1-a) 5.0 dias | horizonte a 50%: 0.6 | a 90%: 3.7
  a 0.90 | escala 2.2942 | memoria 1/(1-a) 10.0 dias | horizonte a 50%: 1.4 | a 90%: 7.9
  a 0.95 | escala 3.2026 | memoria 1/(1-a) 20.0 dias | horizonte a 50%: 2.8 | a 90%: 16.2
  a 0.99 | escala 7.0888 | memoria 1/(1-a) 100.0 dias | horizonte a 50%: 14.3 | a 90%: 82.6


In [2]:
# As duas reconstrucoes, contra as duas formulas.
linhas = []
for a in VALORES_DE_A:
    amostras = [mudanca.ar1(DIAS, np.random.default_rng(SEMENTE + i), a, SIGMA)
                for i in range(SEMENTES)]
    for k in GRADE:
        med_inv, med_otm = [], []
        for x in amostras:
            alvo = x[:-k]
            med_inv.append(np.sqrt(np.mean((esquecimento.reverter(x[k:], a, k) - alvo) ** 2)))
            med_otm.append(np.sqrt(np.mean((a ** k * x[k:] - alvo) ** 2)))
        linhas.append({"a": a, "dias": k,
                       "inverso_medido": float(np.mean(med_inv)),
                       "inverso_formula": esquecimento.erro_do_inverso(a, k, SIGMA),
                       "otimo_medido": float(np.mean(med_otm)),
                       "otimo_formula": esquecimento.erro_do_otimo(a, k, SIGMA)})
tabela = pd.DataFrame(linhas)
resumo = tabela[tabela["dias"].isin((1, 5, 21, 144))].set_index(["a", "dias"])
print(resumo.round(4).to_string())
print()
print("o inverso ja erra no primeiro passo: %s"
      % {a: round(float(tabela[(tabela["a"] == a) & (tabela["dias"] == 1)]["inverso_medido"].iloc[0]), 3)
         for a in VALORES_DE_A})
print("a escala do processo: %s"
      % {a: round(SIGMA / np.sqrt(1 - a ** 2), 3) for a in VALORES_DE_A})


           inverso_medido  inverso_formula  otimo_medido  otimo_formula
a    dias                                                              
0.50 1       2.002500e+00     2.000000e+00        1.0012         1.0000
     5       3.699730e+01     3.693240e+01        1.1562         1.1541
     21      2.425903e+06     2.421583e+06        1.1566         1.1547
     144     2.579179e+43     2.575068e+43        1.1570         1.1547
0.80 1       1.251600e+00     1.250000e+00        1.0013         1.0000
     5       4.817200e+00     4.805400e+00        1.5786         1.5746
     21      1.811319e+02     1.806927e+02        1.6706         1.6666
     144     1.505978e+14     1.502763e+14        1.6710         1.6667
0.90 1       1.112500e+00     1.111100e+00        1.0013         1.0000
     5       3.143600e+00     3.135500e+00        1.8563         1.8515
     21      2.088060e+01     2.084080e+01        2.2845         2.2804
     144     8.913700e+06     8.906393e+06        2.2971        

In [3]:
# O horizonte: onde o erro otimo cruza cada tolerancia declarada.
linhas = []
for a in VALORES_DE_A:
    escala = SIGMA / np.sqrt(1 - a ** 2)
    linha = {"a": a, "escala": escala, "memoria": 1 / (1 - a)}
    for t in TOLERANCIAS:
        curva = tabela[tabela["a"] == a]
        linha["medido_%.2f" % t] = float(np.interp(t * escala, curva["otimo_medido"], curva["dias"]))
        linha["formula_%.2f" % t] = esquecimento.horizonte(a, t)
    linhas.append(linha)
horizontes = pd.DataFrame(linhas).set_index("a")
print(horizontes.round(3).to_string())
print()
print("memoria e horizonte sao medidas diferentes: em a=0,99 a memoria e %d dias e o horizonte a 90%% e %.0f"
      % (1 / (1 - 0.99), esquecimento.horizonte(0.99, 0.9)))


      escala  memoria  medido_0.50  formula_0.50  medido_0.90  formula_0.90
a                                                                          
0.50   1.155      2.0        1.000         0.208        1.321         1.198
0.80   1.667      5.0        1.000         0.645        3.912         3.721
0.90   2.294     10.0        1.422         1.365        7.831         7.881
0.95   3.203     20.0        2.815         2.804       16.898        16.189
0.99   7.089    100.0       14.433        14.312       87.393        82.621

memoria e horizonte sao medidas diferentes: em a=0,99 a memoria e 99 dias e o horizonte a 90% e 83


In [4]:
# O estado guardado com poucos bits: o passado volta igual?
linhas = []
a = 0.9
escala = SIGMA / np.sqrt(1 - a ** 2)
for bits in BITS:
    passo = escala / (2 ** bits)
    ks, erros = [], []
    for k in GRADE:
        v = []
        for x in [mudanca.ar1(DIAS, np.random.default_rng(SEMENTE + i), a, SIGMA)
                  for i in range(6)]:
            guardado = np.round(x / passo) * passo
            v.append(np.sqrt(np.mean((a ** k * guardado[k:] - x[:-k]) ** 2)))
        ks.append(k)
        erros.append(float(np.mean(v)))
    linha_medida = float(np.interp(0.5 * escala, erros, ks))
    linhas.append({"bits": bits, "passo": passo, "horizonte_medido": linha_medida,
                   "horizonte_da_formula": esquecimento.horizonte(a, 0.5)})
memoria_de_bits = pd.DataFrame(linhas).set_index("bits")
print(memoria_de_bits.round(4).to_string())
print()
print("do estado inteiro ao estado de %d bits, o horizonte quase nao se mexe" % BITS[0])


       passo  horizonte_medido  horizonte_da_formula
bits                                                
4     0.1434            1.4289                1.3652
8     0.0090            1.4304                1.3652
16    0.0000            1.4304                1.3652
53    0.0000            1.4304                1.3652

do estado inteiro ao estado de 4 bits, o horizonte quase nao se mexe


In [5]:
# Figura 1: os dois erros contra os dias para tras.
escolhido = 0.9
curva = tabela[tabela["a"] == escolhido]
escala = SIGMA / np.sqrt(1 - escolhido ** 2)
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.plot(curva["dias"], curva["inverso_medido"], marker="o", color="#b03a2e", lw=1.6,
          label="reconstrução inversa (medida)")
eixo.plot(curva["dias"], curva["inverso_formula"], color="#b03a2e", ls=":", lw=1.2,
          label="a fórmula do inverso")
eixo.plot(curva["dias"], curva["otimo_medido"], marker="s", color="#1f4e79", lw=1.6,
          label="reconstrução ótima (medida)")
eixo.plot(curva["dias"], curva["otimo_formula"], color="#1f4e79", ls=":", lw=1.2,
          label="a fórmula do ótimo")
eixo.axhline(escala, color="#555555", ls="--", lw=1.2, label="a escala do processo")
eixo.set_xscale("log"); eixo.set_yscale("log")
eixo.set_xlabel("dias para trás"); eixo.set_ylabel("erro da reconstrução")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E16_o_que_foi_apagado", 1)
plt.close(fig)
print("em %d dias para tras o inverso erra %.1f e o otimo erra %.3f"
      % (GRADE[-1], float(curva[curva["dias"] == GRADE[-1]]["inverso_medido"].iloc[0]),
         float(curva[curva["dias"] == GRADE[-1]]["otimo_medido"].iloc[0])))


em 144 dias para tras o inverso erra 8913700.0 e o otimo erra 2.297


In [6]:
# Figura 2: a memoria e o horizonte de recuperacao, lado a lado.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.plot(horizontes.index, horizontes["memoria"], marker="^", color="#7f8c8d", lw=1.6,
          label="a memória do processo, 1/(1-a)")
for t in TOLERANCIAS:
    eixo.plot(horizontes.index, horizontes["medido_%.2f" % t], marker="o", lw=1.6,
              label="horizonte medido, tolerância de %.0f%%" % (100 * t))
eixo.set_yscale("log")
eixo.set_xlabel("a do processo"); eixo.set_ylabel("dias")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E16_o_que_foi_apagado", 2)
plt.close(fig)
print("horizontes medidos a 90%%: %s" % {a: round(float(horizontes.loc[a, "medido_0.90"]), 1)
                                          for a in horizontes.index})


horizontes medidos a 90%: {0.5: 1.3, 0.8: 3.9, 0.9: 7.8, 0.95: 16.9, 0.99: 87.4}


## Leitura visual das figuras

Figura 1. Os dois eixos estão em escala logarítmica. O horizontal são os dias para trás, de um a pouco mais de cem; o vertical é o erro da reconstrução, com marcas que chegam a dez milhões. A linha cinza tracejada, parada em pouco mais de dois, é a escala do próprio processo, e é ela a régua da figura. A curva azul, a reconstrução ótima, parte do erro de um dia e sobe até encostar nessa linha, onde fica: o passado desbota até o tamanho do processo e para ali. A curva vermelha, a reconstrução inversa, parte do mesmo lugar, pouca coisa acima, cruza a linha cinza em poucos dias e não volta mais a descer; no fim do eixo ela está milhões de vezes acima de tudo o mais que a figura mostra. As duas linhas pontilhadas, que são as fórmulas, caem sobre as curvas medidas em todo o trecho, e é isso que a figura entrega: a explosão tem forma fechada, e não é ruído de simulação. O eixo engana em dois pontos. Numa escala logarítmica nos dois sentidos, a reta aparentemente mansa da vermelha é crescimento exponencial, e o trecho em que ela sai de um e chega a milhão ocupa um punhado de dezenas de dias. E o começo engana ao contrário: nos primeiros dias as duas curvas medidas quase se sobrepõem, e a vermelha ainda não cruzou a régua cinza. O achado visual é esse cruzamento. Antes dele a inversa parece uma alternativa; depois dele a distância entre as duas cresce sem parar, com a azul já acomodada na escala do processo.

Figura 2. O horizontal é o a do processo, de meio a noventa e nove centésimos, em escala linear; o vertical são dias, de um a cem, em escala logarítmica. São três curvas, e a distância entre elas é o que a figura tem a dizer. A cinza é a memória do processo, um sobre um menos a, e sobe de dois dias no começo aos cem do fim. A laranja é o horizonte de recuperação na tolerância larga, noventa por cento, e caminha colada à cinza de ponta a ponta, terminando praticamente sobre ela. A azul é o horizonte na tolerância estreita, cinquenta por cento, e passa a figura inteira embaixo das outras duas: fica reta em um dia desde o começo até oitenta centésimos, e só depois começa a subir, chegando a uma dúzia de dias no extremo. O eixo engana no horizontal, porque é linear numa grandeza que só age no fim: o último décimo, de noventa a noventa e nove centésimos, é onde as três curvas sobem mais de uma ordem de grandeza, e é justamente o trecho mais apertado de ler. Engana também na coincidência, porque a cinza e a laranja andam tão juntas que parecem uma curva só, e são duas medidas diferentes. O achado visual é a azul, a única que se separa o bastante para mostrar que existe mais de uma resposta: metade da escala do processo volta em um dia para qualquer a abaixo de oitenta centésimos, e o horizonte largo acompanha a memória sem quase se distinguir dela.


In [7]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {0.5: "meio", 0.8: "oitenta_centesimos", 0.9: "noventa_centesimos",
         0.95: "noventa_e_cinco_centesimos", 0.99: "noventa_e_nove_centesimos"}
resultado = {
    "apagado_dias": int(DIAS),
    "apagado_sementes": int(SEMENTES),
    "apagado_sigma": float(SIGMA),
    "apagado_dias_maior": int(GRADE[-1]),
    "apagado_bits_menor": int(BITS[0]),
    "apagado_a_do_teste": float(0.9),
    "apagado_horizonte_de_bits": float(memoria_de_bits.loc[BITS[0], "horizonte_medido"]),
    "apagado_horizonte_sem_bits": float(memoria_de_bits.loc[BITS[-1], "horizonte_medido"]),
    "apagado_horizonte_da_formula": float(esquecimento.horizonte(0.9, 0.5)),
}
for a in VALORES_DE_A:
    nome = NOMES[a]
    linha = horizontes.loc[a]
    resultado["apagado_escala_%s" % nome] = float(linha["escala"])
    resultado["apagado_memoria_%s" % nome] = float(linha["memoria"])
    resultado["apagado_horizonte_%s" % nome] = float(linha["medido_0.50"])
    resultado["apagado_horizonte_largo_%s" % nome] = float(linha["medido_0.90"])
    resultado["apagado_inverso_%s" % nome] = float(
        tabela[(tabela["a"] == a) & (tabela["dias"] == 1)]["inverso_medido"].iloc[0])
    resultado["apagado_otimo_%s" % nome] = float(
        tabela[(tabela["a"] == a) & (tabela["dias"] == GRADE[-1])]["otimo_medido"].iloc[0])

caminho = Path("lab/resultados/E16_o_que_foi_apagado.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E16_o_que_foi_apagado.json gravado | 39 grandezas
